# Disease Outbreak Prediction - End-to-End Pipeline

This notebook provides a complete pipeline for predicting disease outbreaks using historical data. It covers:
1. Data Loading and Preprocessing
2. Feature Engineering
3. Model Training (Random Forest, Gradient Boosting, ARIMA, Prophet)
4. Evaluation and Inference (Best Model Selection)

In [ ]:
%pip install plotly prophet statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import os
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 1. Data Preprocessing

Defining functions to load and clean the dataset.

In [ ]:
import pandas as pd
import numpy as np

def load_and_preprocess(filepath):
    """
    Loads disease case data and performs initial cleaning.
    """
    df = pd.read_csv(filepath)
    
    # Handle missing values
    df = df.dropna(subset=['week', 'cases', 'state'])
    
    # Ensure cases is numeric
    df['cases'] = pd.to_numeric(df['cases'], errors='coerce')
    df = df.dropna(subset=['cases'])
    df['cases'] = df['cases'].astype(int)
    
    # Convert week (YYYYWW) to datetime
    # Adding '0' for Sunday of that week as a convention
    df['date'] = pd.to_datetime(df['week'].astype(str) + '0', format='%Y%U%w')
    
    # Sort chronologically
    df = df.sort_values(['state', 'date'])
    
    # Remove duplicates
    df = df.drop_duplicates()
    
    # Fill missing cases with 0 or forward fill? Let's use 0 for disease cases if missing
    df['cases'] = df['cases'].fillna(0)
    
    return df


## 2. Feature Engineering

Creating lag features, rolling averages, and seasonal indicators.

In [ ]:
import pandas as pd
import numpy as np

def generate_features(df):
    """
    Generates time-series and seasonal features for outbreak prediction.
    """
    # Group by state to ensure features are per-region
    df_sorted = df.sort_values(['state', 'date'])
    
    # 1. Lag Features (1, 2, 4 weeks)
    df_sorted['cases_lag1'] = df_sorted.groupby('state')['cases'].shift(1)
    df_sorted['cases_lag2'] = df_sorted.groupby('state')['cases'].shift(2)
    df_sorted['cases_lag4'] = df_sorted.groupby('state')['cases'].shift(4)
    
    # 2. Rolling Averages (4 and 12 weeks)
    df_sorted['rolling_avg_4'] = df_sorted.groupby('state')['cases'].transform(lambda x: x.rolling(window=4).mean())
    df_sorted['rolling_avg_12'] = df_sorted.groupby('state')['cases'].transform(lambda x: x.rolling(window=12).mean())
    
    # 3. Seasonal Indicators
    df_sorted['month'] = df_sorted['date'].dt.month
    df_sorted['year'] = df_sorted['date'].dt.year
    df_sorted['quarter'] = df_sorted['date'].dt.quarter
    
    # 4. Growth Rate (Percentage change week-over-week)
    df_sorted['growth_rate'] = df_sorted.groupby('state')['cases'].pct_change().replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # Drop rows with NaNs resulting from lags/rolling
    df_sorted = df_sorted.dropna()
    
    return df_sorted


## 3. Model Training (ML Models)

Defining the training logic for Random Forest and Gradient Boosting models.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle
import os
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet
import warnings

warnings.filterwarnings('ignore')

def train_arima_model(train, test):
    # ARIMA requires a univariate series
    history = list(train['cases'])
    test_cases = list(test['cases'])
    predictions = []
    
    # Rolling Forecast
    print(f"    Running ARIMA Rolling Forecast on {len(test_cases)} points...")
    # Pre-train model once to get parameters? Or re-train every step?
    # Re-training every step is too slow. 
    # Strategy: Fit once, then Append new observations.
    try:
        model = ARIMA(history, order=(5,1,0))
        model_fit = model.fit()
        
        for t in range(len(test_cases)):
            # Forecast 1 step
            output = model_fit.forecast()
            yhat = output[0]
            predictions.append(yhat)
            
            # Observe actual
            obs = test_cases[t]
            history.append(obs)
            
            # Update model with new observation (append, don't re-fit fully for speed)
            model_fit = model_fit.append([obs], refit=False)
            
    except Exception as e:
        print(f"ARIMA training failed: {e}")
        predictions = [0] * len(test)
        
    return predictions

def train_prophet_model(train, test):
    # Prophet requires 'ds' and 'y' columns
    # Rolling forecast for Prophet is tricky/slow as it usually needs re-fitting.
    # But we can try to simulate it or at least fit on ALL available data up to split.
    # Actually, for a fair comparison with ML "One Step Ahead", we should retrain or use a big horizon?
    # Prophet is designed for horizons. But let's try to improve it.
    # Re-fitting 52 times is slow.
    # Compromise: Fit on training. Predict.
    # To truly fix "low accuracy" compared to ML (which sees t-1), 
    # we'd need to assume Prophet sees t-1.
    # But Prophet generates a 'trend'. 
    # Let's keep Prophet as is (Horizon Forecast) because that's its strength,
    # OR we can try to provide regressors?
    # Let's leave Prophet as 52-week forecast (standard use) but acknowledge it's harder.
    # OR: we can try to re-fit every 12 weeks?
    # Let's stick to the standard forecast for Prophet but ensure data is clean.
    
    prophet_train = train[['date', 'cases']].rename(columns={'date': 'ds', 'cases': 'y'})
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    m.fit(prophet_train)
    
    future = pd.DataFrame(test['date']).rename(columns={'date': 'ds'})
    forecast = m.predict(future)
    return forecast['yhat'].values


def calculate_accuracy(y_true, y_pred):
    """
    Calculates accuracy using 100 - SMAPE (Symmetric Mean Absolute Percentage Error).
    Includes a small constant to handle low values (0 vs 1) gracefully.
    """
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Baseline granularity constant (handles near-zero values gracefully)
    C = 1.0 
    denominator = np.abs(y_true) + np.abs(y_pred) + C
    diff = np.abs(y_true - y_pred)
    
    smape = 2.0 * diff / denominator
    mean_smape = np.mean(smape)
    
    # Scale: SMAPE stays [0, 2], so we normalize by /2 to get [0, 1] error scale.
    accuracy = 100 * (1 - (mean_smape / 2.0))
    return max(0.0, accuracy)

def train_models(df, disease_name):
    # Features and Target
    features = ['cases_lag1', 'cases_lag2', 'cases_lag4', 'rolling_avg_4', 'rolling_avg_12', 'month', 'year', 'quarter', 'growth_rate']
    target = 'cases'
    
    max_date = df['date'].max()
    split_date = max_date - pd.Timedelta(weeks=52)
    
    train = df[df['date'] <= split_date]
    test = df[df['date'] > split_date]
    
    X_train, y_train = train[features], train[target]
    X_test, y_test = test[features], test[target]
    
    # Aggregated National Target for Comparison
    test_ts = test.groupby('date')['cases'].sum().reset_index()
    y_test_national = test_ts['cases'].values
    
    # Models
    ml_models = {
        "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42)
    }
    
    results = {}
    os.makedirs('models', exist_ok=True)
    
    # 1. Train ML Models
    for name, model in ml_models.items():
        print(f"  Training {name}...")
        model.fit(X_train, y_train)
        
        # Predict at state level
        test_with_preds = test.copy()
        test_with_preds['preds'] = model.predict(X_test)
        
        # AGGREGATE to National level for metric calculation
        national_preds = test_with_preds.groupby('date')['preds'].sum().values
        
        mae = mean_absolute_error(y_test_national, national_preds)
        rmse = np.sqrt(mean_squared_error(y_test_national, national_preds))
        accuracy = calculate_accuracy(y_test_national, national_preds)
        
        results[name] = {"MAE": mae, "RMSE": rmse, "Accuracy": accuracy}
        
        # Save model
        model_filename = f'models/{disease_name.lower()}_{name.lower().replace(" ", "_")}.pkl'
        with open(model_filename, 'wb') as f:
            pickle.dump(model, f)

    # 2. Train Time Series Models (on already aggregated data)
    train_ts = train.groupby('date')['cases'].sum().reset_index()
    
    # Train ARIMA
    print("  Training ARIMA...")
    try:
        arima_preds = train_arima_model(train_ts, test_ts)
        
        arima_mae = mean_absolute_error(y_test_national, arima_preds)
        arima_rmse = np.sqrt(mean_squared_error(y_test_national, arima_preds))
        arima_acc = calculate_accuracy(y_test_national, arima_preds)
        results["ARIMA"] = {"MAE": arima_mae, "RMSE": arima_rmse, "Accuracy": arima_acc}
        
        # Save placeholder for consistency (logic in app.py handles loading)
        with open(f'models/{disease_name.lower()}_arima.pkl', 'wb') as f:
            pickle.dump(None, f) 
    except Exception as e:
        print(f"  ARIMA failed: {e}")
        results["ARIMA"] = {"MAE": 99999, "RMSE": 99999, "Accuracy": 0}

    # Train Prophet
    print("  Training Prophet...")
    try:
        prophet_preds = train_prophet_model(train_ts, test_ts)
        
        prophet_mae = mean_absolute_error(y_test_national, prophet_preds)
        prophet_rmse = np.sqrt(mean_squared_error(y_test_national, prophet_preds))
        prophet_acc = calculate_accuracy(y_test_national, prophet_preds)
        results["Prophet"] = {"MAE": prophet_mae, "RMSE": prophet_rmse, "Accuracy": prophet_acc}
        
        # Save Prophet
        with open(f'models/{disease_name.lower()}_prophet.pkl', 'wb') as f:
             # Just to satisfy loading logic, we save the dummy or retrain as needed
             pickle.dump(None, f) 
    except Exception as e:
         print(f"  Prophet failed: {e}")
         results["Prophet"] = {"MAE": 99999, "RMSE": 99999, "Accuracy": 0}
            
    return results, test, features


## 4. Advanced Models (ARIMA & Prophet)

Defining training logic for time-series specific models.

In [ ]:

def train_arima_model(train, test):
    # ARIMA requires a univariate series
    history = list(train['cases'])
    test_cases = list(test['cases'])
    predictions = []
    
    # Rolling Forecast
    print(f"    Running ARIMA Rolling Forecast on {len(test_cases)} points...")
    try:
        model = ARIMA(history, order=(5,1,0))
        model_fit = model.fit()
        
        for t in range(len(test_cases)):
            # Forecast 1 step
            output = model_fit.forecast()
            yhat = output[0]
            predictions.append(yhat)
            
            # Observe actual
            obs = test_cases[t]
            history.append(obs)
            
            # Update model with new observation (append, don't re-fit fully for speed)
            model_fit = model_fit.append([obs], refit=False)
            
    except Exception as e:
        print(f"ARIMA training failed: {e}")
        predictions = [0] * len(test)

    return predictions

def train_prophet_model(train, test):
    # Prophet requires 'ds' and 'y' columns
    prophet_train = train[['date', 'cases']].rename(columns={'date': 'ds', 'cases': 'y'})
    
    m = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    m.fit(prophet_train)
    
    future = pd.DataFrame(test['date']).rename(columns={'date': 'ds'})
    forecast = m.predict(future)
    
    return forecast['yhat'].values


## 5. Prediction & Visualization

Functions for forecasting, severity classification, and plotting trends.

In [ ]:
import pandas as pd
import numpy as np
import pickle

def classify_severity(cases, thresholds):
    """
    Classifies severity based on thresholds.
    - Low: <= Median
    - Medium: Median < Cases < P90
    - High: >= P90
    """
    if cases >= thresholds['med_high']:
        return 'High'
    elif cases > thresholds['low_med']:
        return 'Medium'
    else:
        return 'Low'

def get_thresholds(df):
    """
    Calculates threshold values based on historical data percentiles.
    """
    return {
        'low_med': df['cases'].median(),
        'med_high': df['cases'].quantile(0.90)
    }

def forecast_future(model_path, current_data, features, weeks=12):
    """
    Simple forecasting by iteratively predicting future weeks using the model.
    """
    with open(model_path, 'rb') as f:
        model = pickle.load(f)
    
    forecast_data = current_data.tail(1).copy()
    predictions = []
    
    # Simple recursive forecast: use predictions as next lag features
    # For demonstration, we'll just predict on a provided future/test set
    return model.predict(current_data[features])


import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

def plot_trends(results_df, state_name, disease_name, save_path=None):
    """
    Plots actual vs predicted trends for a specific state and disease.
    """
    sns.set(style="whitegrid")
    plt.figure(figsize=(14, 7))
    
    state_data = results_df[results_df['state'] == state_name]
    
    plt.plot(state_data['date'], state_data['cases'], label='Actual Cases', color='blue', linewidth=2, alpha=0.6)
    plt.plot(state_data['date'], state_data['predicted_cases'], label='Predicted Cases', color='orange', linestyle='--', linewidth=2)
    
    # Highlight severity
    high_severity = state_data[state_data['severity'] == 'High']
    plt.scatter(high_severity['date'], high_severity['cases'], color='red', label='Outbreak (High Severity)', zorder=5)
    
    plt.title(f'{disease_name} Outbreak Prediction - {state_name}', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Case Count', fontsize=12)
    plt.legend()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f"Plot saved to {save_path}")
    plt.close()


In [ ]:

# === MAIN EXECUTION WITH KAGGLE DATASET PATH ===

DATA_DIR = "e:\TRIZEN\DOP\backend\datasets" # Update this on Kaggle!
output_results = {}

if not os.path.exists(DATA_DIR):
    print(f"Warning: Dataset directory not found at {DATA_DIR}. Please upload data.")
    files = []
else:
    files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    print(f"Found {len(files)} datasets: {files}")
    
for filename in files:
    disease_name = filename.replace('.csv', '').capitalize()
    file_path = os.path.join(DATA_DIR, filename)
    
    print(f"\n{'='*60}")
    print(f"Processing: {disease_name}")
    print(f"{'='*60}")
    
    # 1. Load Data
    df = load_and_preprocess(file_path)
    
    # 2. Feature Engineering
    df = generate_features(df)
    
    # ... (inside loop) ...
    
    def calculate_accuracy(y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        C = 1.0 # Baseline granularity constant
        denominator = np.abs(y_true) + np.abs(y_pred) + C
        diff = np.abs(y_true - y_pred)
        smape = 2.0 * diff / denominator
        mean_smape = np.mean(smape)
        return max(0.0, 100 * (1 - (mean_smape / 2.0)))

    # 3. Model Training & Comparison
    print(f"Training ALL models for {disease_name}...")
    
    # Aggregated National Target for Comparison
    test_ts = test_data.groupby('date')['cases'].sum().reset_index()
    y_test_national = test_ts['cases'].values
    
    # A. ML Models (RF, GB)
    # Note: train_models now returns metrics calculated on NATIONAL AGGREGATES
    ml_results, test_data, features = train_models(df, disease_name)
    
    # Target values for comparison (National Sum)
    # This must match what train_models used: test_data (last 52 weeks)
    ts_test_data = test_data.groupby('date')['cases'].sum().reset_index()
    y_test_national = ts_test_data['cases'].values

    # B. ARIMA
    print("Training ARIMA...")
    # Prepare historical training data (aggregated)
    split_date = test_data['date'].min()
    train_data_ts = df[df['date'] < split_date].groupby('date')['cases'].sum().reset_index()
    
    arima_preds = train_arima_model(train_data_ts, ts_test_data)
    arima_mae = mean_absolute_error(y_test_national, arima_preds)
    arima_rmse = np.sqrt(mean_squared_error(y_test_national, arima_preds))
    arima_acc = calculate_accuracy(y_test_national, arima_preds)
    ml_results["ARIMA"] = {"MAE": arima_mae, "RMSE": arima_rmse, "Accuracy": arima_acc} 
    
    # C. Prophet
    print("Training Prophet...")
    prophet_preds = train_prophet_model(train_data_ts, ts_test_data)
    prophet_mae = mean_absolute_error(y_test_national, prophet_preds)
    prophet_rmse = np.sqrt(mean_squared_error(y_test_national, prophet_preds))
    prophet_acc = calculate_accuracy(y_test_national, prophet_preds)
    ml_results["Prophet"] = {"MAE": prophet_mae, "RMSE": prophet_rmse, "Accuracy": prophet_acc}
    
    # 4. Compare and Select Best Model
    print(f"\n{disease_name} Model Performance:")
    best_model_name = ""
    best_accuracy = -1
    
    metrics_data = []

    for name, metric in ml_results.items():
        acc_val = metric.get('Accuracy', 0)
        print(f"{name}: MAE = {metric['MAE']:.2f}, RMSE = {metric['RMSE']:.2f}, Accuracy = {acc_val:.2f}%")
        metrics_data.append({"Model": name, "RMSE": metric['RMSE'], "MAE": metric['MAE'], "Accuracy": acc_val})
        
        if acc_val > best_accuracy:
            best_accuracy = acc_val
            best_model_name = name
            
    print(f"\n>> Best Model Selected: {best_model_name} (Accuracy={best_accuracy:.2f}%)")
    
    # --- New: Model Comparison Visualization ---
    metrics_df = pd.DataFrame(metrics_data)
    
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.barplot(x='Model', y='RMSE', data=metrics_df, palette='viridis')
    plt.title(f'RMSE (Lower is Better)')
    
    plt.subplot(1, 2, 2)
    sns.barplot(x='Model', y='Accuracy', data=metrics_df, palette='magma')
    plt.title(f'Accuracy (Higher is Better)')
    plt.show()
    
    print("\nModel Comparison Table:")
    print(metrics_df.to_markdown(index=False) if hasattr(metrics_df, 'to_markdown') else metrics_df)
    # -------------------------------------------

    # 5. Visualization (Using Best Model Predictions)
    # We need to plot the predictions of the best model.
    # We need to map the predictions back to the 'test_data' dataframe structure for the plot_trends function
    
    if best_model_name in ["Random Forest", "Gradient Boosting"]:
        with open(f'models/{disease_name.lower()}_{best_model_name.lower().replace(" ", "_")}.pkl', 'rb') as f:
            loaded_model = pickle.load(f)
        test_data['predicted_cases'] = loaded_model.predict(test_data[features])
        
    elif best_model_name == "ARIMA":
        # ARIMA predicts aggregate, we need to distribute or just plot aggregate?
        # plot_trends plots per state. ARIMA/Prophet here were trained on AGGREGATE.
        # Limitation: Advanced models in this simple loop are global.
        # For visualization consistency, let's map aggregate predictions to the first state or aggregated view.
        # Better: Plot aggregate trends for ARIMA/Prophet or stick to ML models for State-wise.
        # FIX: Let's assign the aggregate prediction proportional to state presence or just copy for viz if 1 state.
        # For simplicity: Use the predictions directly if simple, else warn.
        print("Note: ARIMA/Prophet predictions are aggregated. Plotting aggregate comparison.")
        test_data = ts_test_data.copy() # Switch to using the aggregated test set for plotting
        test_data['state'] = 'Aggregated' # Dummy state
        test_data['predicted_cases'] = arima_preds
        
    elif best_model_name == "Prophet":
        print("Note: ARIMA/Prophet predictions are aggregated. Plotting aggregate comparison.")
        test_data = ts_test_data.copy()
        test_data['state'] = 'Aggregated'
        test_data['predicted_cases'] = prophet_preds

    # Ensure required columns for plot_trends
    if 'severity' not in test_data.columns:
         thresholds = get_thresholds(df) # global thresholds
         test_data['severity'] = test_data['predicted_cases'].apply(lambda x: classify_severity(x, thresholds))

    # Plot
    
print(f"Visualizing trends for: {disease_name}")
    # If using ML models, test_data has multiple states. default plot_trends takes one state.
    state_to_plot = test_data['state'].iloc[0]
    plot_trends(test_data, state_to_plot, disease_name)
    plt.show()

print("\nProcessing Complete for all datasets.")

# 6. Zip Models for Download
import shutil
shutil.make_archive('trained_models', 'zip', 'models')
print("Models zipped to trained_models.zip. You can download this file from the Output section.")
